In [1]:
# Set project paths.
from pathlib import Path
import os
import sys

def find_project_root():
    current = Path.cwd()

    for folder in [current] + list(current.parents):
        if (folder / "Data").exists() and (folder / "Notebooks").exists():
            return folder

    raise FileNotFoundError("Could not find project root. Make sure Data and Notebooks folders exist.")

project_folder = find_project_root()
notebook_folder = project_folder / "Notebooks"

os.chdir(project_folder)

print("Project folder:", project_folder)
print("Notebook folder:", notebook_folder)

Project folder: /Users/mac/Library/CloudStorage/OneDrive-UniversityofKeele/Dissertation/send-ev-project
Notebook folder: /Users/mac/Library/CloudStorage/OneDrive-UniversityofKeele/Dissertation/send-ev-project/Notebooks


In [2]:
# Import packages and helpers.
os.chdir(notebook_folder)

import import_ipynb
import pandas as pd
import plotly.express as px

from Data_Loader import DataLoader

os.chdir(notebook_folder)

from Training_Models import PVModel

os.chdir(project_folder)

print("Current folder:", os.getcwd())

Project folder: /Users/mac/Library/CloudStorage/OneDrive-UniversityofKeele/Dissertation/send-ev-project
Notebook folder: /Users/mac/Library/CloudStorage/OneDrive-UniversityofKeele/Dissertation/send-ev-project/Notebooks
                     air_temp  gti  surface_pressure  snow_depth  \
DateTime                                                           
2023-01-01 00:05:00         9    0             978.0         0.0   
2023-01-01 00:10:00         9    0             978.1         0.0   
2023-01-01 00:15:00         9    0             978.2         0.0   
2023-01-01 00:20:00         9    0             978.2         0.0   
2023-01-01 00:25:00         9    0             978.3         0.0   

                     cloud_opacity  ghi  clearsky_gti  wind_speed_100m  \
DateTime                                                                 
2023-01-01 00:05:00           33.8    0             0             11.0   
2023-01-01 00:10:00           26.3    0             0             11.2   
2023-01-

In [3]:
# Load data.
solcast, deop, expected = DataLoader.load_training_data()
deop_2022, solcast_2022, expected_2022 = DataLoader.load_testing_data()

print(deop.head())
print(solcast.head())
print(deop_2022.head())

                     power-con-ave  power-gen-wt-ave  power-gen-pv-ave
DateTime                                                              
2023-01-01 00:05:00       1317.487           573.029               0.0
2023-01-01 00:10:00       1319.056           612.280               0.0
2023-01-01 00:15:00       1338.842           652.990               0.0
2023-01-01 00:20:00       1329.313           813.526               0.0
2023-01-01 00:25:00       1338.366           643.773               0.0
                     air_temp  gti  surface_pressure  snow_depth  \
DateTime                                                           
2023-01-01 00:05:00         9    0             978.0         0.0   
2023-01-01 00:10:00         9    0             978.1         0.0   
2023-01-01 00:15:00         9    0             978.2         0.0   
2023-01-01 00:20:00         9    0             978.2         0.0   
2023-01-01 00:25:00         9    0             978.3         0.0   

                     cloud

In [4]:
# Check date ranges.
print("2023 DEOP:", deop.index.min(), "to", deop.index.max())
print("2023 Solcast:", solcast.index.min(), "to", solcast.index.max())

print("2022 DEOP:", deop_2022.index.min(), "to", deop_2022.index.max())
print("2022 Solcast:", solcast_2022.index.min(), "to", solcast_2022.index.max())

2023 DEOP: 2023-01-01 00:05:00 to 2023-12-31 23:55:00
2023 Solcast: 2023-01-01 00:05:00 to 2023-12-31 23:55:00
2022 DEOP: 2022-03-01 00:00:00 to 2022-12-31 23:55:00
2022 Solcast: 2022-01-01 00:05:00 to 2022-12-31 23:55:00


In [5]:
# Choose solar features.
pv_features = [
    "solar_potential",
    "hour_sin",
    "hour_cos",
    "cloud_opacity_target",
    "lag_2d",
    "lag_3d",
    "lag_14d",
    "power_volatility_1d",
    "power_trend_1d",
    "month",
    "day_of_month",
    "gti_volatility_1d",
    "yesterday_peak",
    "lag_1d",
    "lag_7d",
]

In [6]:
# Train solar model.
pv = PVModel(features=pv_features)

pv_preds, pv_model = pv.train_and_test(
    deop,
    solcast,
    deop_2022,
    solcast_2022,
    days_ahead=1,
)

Training new PV model
5-Minute R2 Score: 0.8181 | MAE: 177.81 kW | MBE: -12.68 kW
Hourly R2 Score:   0.8813 | MAE: 138.03 kW
Daily R2 Score:    0.8538 | MAE: 91.65 kW


In [7]:
# Build results table.
solar_results = pd.DataFrame({
    "actual": deop_2022.loc[pv_preds.index, "power-gen-pv-ave"],
    "predicted": pv_preds,
})

solar_results.head()

,actual,predicted
DateTime,,
2022-03-15 00:00:00,0.0,0.0
2022-03-15 00:05:00,0.0,0.0
2022-03-15 00:10:00,0.0,0.0
2022-03-15 00:15:00,0.0,0.0
2022-03-15 00:20:00,0.0,0.0


In [8]:
# Plot actual vs predicted.
hourly_results = solar_results.resample("1h").mean()

fig = px.line(
    hourly_results,
    y=["actual", "predicted"],
    title="Solar PV: Actual vs Predicted",
    labels={"value": "Power [kW]", "DateTime": "Time"},
)

fig.show()

In [9]:
# Calculate monthly scores.
from sklearn.metrics import r2_score, mean_absolute_error

monthly_scores = []

for month, group in solar_results.groupby(solar_results.index.month):
    r2 = r2_score(group["actual"], group["predicted"])
    mae = mean_absolute_error(group["actual"], group["predicted"])

    monthly_scores.append({
        "month": month,
        "r2": r2,
        "mae": mae,
    })

monthly_scores = pd.DataFrame(monthly_scores)
monthly_scores

,month,r2,mae
0,3,0.870577,184.651448
1,4,0.817300,205.851755
2,5,0.839148,204.899968
3,6,0.766424,253.514457
4,7,0.783416,232.396595
5,8,0.780834,228.523870
6,9,0.816047,172.563515
7,10,0.799624,151.568562
8,11,0.783017,64.238972
9,12,0.708382,82.448326


In [10]:
# Plot monthly scores.
fig = px.bar(
    monthly_scores,
    x="month",
    y="r2",
    title="Solar Model Monthly R2 Score",
    labels={"month": "Month", "r2": "R2 Score"},
)

fig.show()

In [11]:
# Run forecast horizons.
pv_predictions = {}

for i in range(1, 15):
    print("Running", i, "day forecast")

    preds, _ = pv.train_and_test(
        deop,
        solcast,
        deop_2022,
        solcast_2022,
        days_ahead=i,
    )

    pv_predictions[f"{i}-Day Forecast"] = preds

Running 1 day forecast
Loading PV model from Models/pv_model_feats_edbb702a.json
5-Minute R2 Score: 0.8181 | MAE: 177.81 kW | MBE: -12.68 kW
Hourly R2 Score:   0.8813 | MAE: 138.03 kW
Daily R2 Score:    0.8538 | MAE: 91.65 kW
Running 2 day forecast
Loading PV model from Models/pv_model_feats_edbb702a.json
5-Minute R2 Score: 0.8096 | MAE: 180.43 kW | MBE: -10.68 kW
Hourly R2 Score:   0.8726 | MAE: 140.64 kW
Daily R2 Score:    0.8416 | MAE: 91.98 kW
Running 3 day forecast
Loading PV model from Models/pv_model_feats_edbb702a.json
5-Minute R2 Score: 0.8112 | MAE: 179.91 kW | MBE: -13.99 kW
Hourly R2 Score:   0.8746 | MAE: 139.63 kW
Daily R2 Score:    0.8427 | MAE: 91.50 kW
Running 4 day forecast
Loading PV model from Models/pv_model_feats_edbb702a.json
5-Minute R2 Score: 0.8111 | MAE: 180.64 kW | MBE: -11.56 kW
Hourly R2 Score:   0.8741 | MAE: 140.71 kW
Daily R2 Score:    0.8473 | MAE: 90.36 kW
Running 5 day forecast
Loading PV model from Models/pv_model_feats_edbb702a.json
5-Minute R2 Sco

In [12]:
# Compare horizons.
forecast_scores = []

for name, preds in pv_predictions.items():
    actual = deop_2022.loc[preds.index, "power-gen-pv-ave"]

    r2 = r2_score(actual, preds)
    mae = mean_absolute_error(actual, preds)

    forecast_scores.append({
        "forecast": name,
        "r2": r2,
        "mae": mae,
    })

forecast_scores = pd.DataFrame(forecast_scores)
forecast_scores

,forecast,r2,mae
0,1-Day Forecast,0.818121,177.805111
1,2-Day Forecast,0.809613,180.431304
2,3-Day Forecast,0.811202,179.906738
3,4-Day Forecast,0.811134,180.641353
4,5-Day Forecast,0.810992,180.787467
5,6-Day Forecast,0.811100,182.099553
6,7-Day Forecast,0.813762,180.881073
7,8-Day Forecast,0.809976,182.601715
8,9-Day Forecast,0.810856,182.627302
9,10-Day Forecast,0.808544,183.405292


In [13]:
# Plot horizon scores.
fig = px.bar(
    forecast_scores,
    x="forecast",
    y="r2",
    title="Solar Model Performance by Forecast Horizon",
    labels={"forecast": "Forecast Horizon", "r2": "R2 Score"},
)

fig.show()

In [14]:
# Plot feature importance.
importance_values = pv_model.feature_importances_

feature_importance = pd.Series(
    importance_values,
    index=pv_features,
).sort_values()

fig = px.bar(
    x=feature_importance.values,
    y=feature_importance.index,
    orientation="h",
    title="Solar Model Feature Importance",
    labels={"x": "Importance", "y": "Feature"},
)

fig.show()